In [1]:
import numpy as np
import torch
import torch.nn as nn

import itertools
import tqdm

from pathlib import Path
from torch.utils.data import DataLoader

from src.data import HypersphereDynamicsDataset, DynamicsDatasetMode
from src.model import DynamicsModel

In [2]:
torch.set_default_dtype(torch.float64)
device_cuda = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device_cpu = torch.device("cpu")
device_cuda

device(type='cuda')

In [3]:
HYPERSPHERE_DIM = 4
HYPERSPHERE_RADII = [1.0, 0.5, 2.0]

DATA_MODES = [
    DynamicsDatasetMode.SINGLE_COORD_CHART,
    DynamicsDatasetMode.MULTI_COORD_CHART,
    DynamicsDatasetMode.MULTI_COORD_CHART_WITH_VALIDITY,
    DynamicsDatasetMode.EXTRINSIC_COORD
]

TRAINING_EPOCHS = 1000
TRAINING_LR = [1e-4] # [1E-4, 2.5E-4]  # , 2.5E-4]
TRAINING_BATCH_SIZE = [128] # [128, 256]
TRAINING_MODEL_PARAMS = [
    # num_hidden_layers, num_nodes_in_hidden_layer, activation_fn
    # (5, 40, nn.LeakyReLU),
    (5, 60, nn.LeakyReLU),
    # (5, 80, nn.LeakyReLU),

    # (6, 40, nn.LeakyReLU),
    (6, 60, nn.LeakyReLU),
    # (6, 80, nn.LeakyReLU),

    # (7, 40, nn.LeakyReLU),
    (7, 60, nn.LeakyReLU),
    # (7, 80, nn.LeakyReLU),
]

DYNAMICS_DATA_DIR = Path("../../../data/unforced_dynamics")
TRAINING_RESULTS_DIR = Path("../../../data/training")

DYNAMICS_SUBDIR_FORMAT = "dim_{dim}/radius_{radius}"
TRAINING_SUBDIR_FORMAT = "dim_{dim}/radius_{radius}"


In [4]:
def train_loop(
        dataloader: DataLoader,
        model: DynamicsModel,
        loss_fn,
        optimizer: torch.optim.Optimizer, ) -> torch.Tensor:
    model.train()

    batch_losses = torch.zeros(len(dataloader), device=device_cpu)

    for batch_idx, (inputs, outputs) in enumerate(dataloader):
        inputs = inputs.to(device_cuda, non_blocking=True)
        outputs = outputs.to(device_cuda, non_blocking=True)

        pred_outputs = model(inputs)
        loss: torch.Tensor = loss_fn(pred_outputs, outputs)

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        batch_losses[batch_idx] = loss.item()

    return batch_losses


def valid_loop(
        dataloader: DataLoader,
        model: DynamicsModel,
        loss_fn,
) -> torch.Tensor:
    model.eval()

    batch_losses = torch.zeros(len(dataloader), device=device_cpu)

    for batch_idx, (inputs, outputs) in enumerate(dataloader):
        inputs = inputs.to(device_cuda, non_blocking=True)
        outputs = outputs.to(device_cuda, non_blocking=True)

        pred_outputs = model(inputs)
        loss: torch.Tensor = loss_fn(pred_outputs, outputs)

        batch_losses[batch_idx] = loss.item()

    return batch_losses


In [5]:
cfgs = list(itertools.product(
    DATA_MODES, HYPERSPHERE_RADII, TRAINING_MODEL_PARAMS, TRAINING_LR, TRAINING_BATCH_SIZE
))
for i, (mode, radius, (num_hidden_layers, num_nodes_in_hidden_layer, activation_fn), lr, batch_size) in enumerate(cfgs):
    print(f"cfg: {i}/{len(cfgs)}")

    # sets up the training/validation data
    training_dir = DYNAMICS_DATA_DIR / DYNAMICS_SUBDIR_FORMAT.format(dim=HYPERSPHERE_DIM, radius=radius)
    training_data, validation_data = HypersphereDynamicsDataset.load(
        dir_path=training_dir,
        n=HYPERSPHERE_DIM,
        radius=radius,
        mode=mode,
        device=device_cpu, )

    training_dataloader = DataLoader(training_data, batch_size=batch_size, shuffle=True, num_workers=3,
                                     pin_memory=False, persistent_workers=True)
    validation_dataloader = DataLoader(validation_data, batch_size=batch_size, shuffle=True, num_workers=3,
                                       pin_memory=False, persistent_workers=True)

    # sets up the parameterized model
    num_input_features = training_data.num_input_features
    num_output_features = training_data.num_output_features

    model = DynamicsModel(
        num_input_features=num_input_features,
        num_output_features=num_output_features,
        num_hidden_layers=num_hidden_layers,
        num_nodes_in_hidden_layer=num_nodes_in_hidden_layer,
        activation_fn=activation_fn,
    ).to(device_cuda)

    loss_fn = nn.MSELoss()
    optimizer = torch.optim.Adam(params=model.parameters(), lr=lr)

    # pre-allocates space for the loss histories (epochs, num_batches)
    train_batch_loss_hist = torch.zeros((TRAINING_EPOCHS, len(training_dataloader)), device=device_cpu)
    valid_batch_loss_hist = torch.zeros((TRAINING_EPOCHS, len(validation_dataloader)), device=device_cpu)

    # performs the actual training
    pbar = tqdm.tqdm(range(TRAINING_EPOCHS), desc="Training")
    for epoch in pbar:
        train_batch_losses = train_loop(training_dataloader, model, loss_fn, optimizer)
        valid_batch_losses = valid_loop(validation_dataloader, model, loss_fn)

        train_batch_loss_hist[epoch, :] = train_batch_losses
        valid_batch_loss_hist[epoch, :] = valid_batch_losses

        pbar.set_postfix(
            avg_train_batch_loss=torch.mean(train_batch_losses),
            avg_valid_batch_loss=torch.mean(valid_batch_losses),
        )

    # outputs this data for processing later
    train_batch_loss_hist_numpy = train_batch_loss_hist.numpy()
    valid_batch_loss_hist_numpy = valid_batch_loss_hist.numpy()

    dir_path = TRAINING_RESULTS_DIR / TRAINING_SUBDIR_FORMAT.format(dim=HYPERSPHERE_DIM, radius=radius)

    results_filename = f"train_results_mode_{mode}_npl_{num_nodes_in_hidden_layer}_layers_{num_hidden_layers}_bs_{batch_size}_lr_{lr}.npz"
    model_filename = f"train_results_mode_{mode}_npl_{num_nodes_in_hidden_layer}_layers_{num_hidden_layers}_bs_{batch_size}_lr_{lr}.pt"

    torch.save(model.state_dict(), dir_path / model_filename)
    np.savez(
        dir_path / results_filename,
        n=HYPERSPHERE_DIM,
        radius=radius,
        batch_size=batch_size,
        lr=lr,
        num_hidden_layers=num_hidden_layers,
        num_nodes_in_hidden_layer=num_nodes_in_hidden_layer,
        train_batch_loss_hist_numpy=train_batch_loss_hist_numpy,
        valid_batch_loss_hist_numpy=valid_batch_loss_hist_numpy,
        mode=mode,
    )

cfg: 0/36


Training: 100%|██████████| 1000/1000 [37:36<00:00,  2.26s/it, avg_train_batch_loss=tensor(0.0057), avg_valid_batch_loss=tensor(0.0103)]


cfg: 1/36


Training: 100%|██████████| 1000/1000 [41:29<00:00,  2.49s/it, avg_train_batch_loss=tensor(0.0055), avg_valid_batch_loss=tensor(0.0109)]


cfg: 2/36


...

Training: 100%|██████████| 1000/1000 [37:49<00:00,  2.27s/it, avg_train_batch_loss=tensor(1.6878e-06), avg_valid_batch_loss=tensor(5.1927e-06)]


cfg: 34/36


Training: 100%|██████████| 1000/1000 [42:59<00:00,  2.58s/it, avg_train_batch_loss=tensor(2.6290e-06), avg_valid_batch_loss=tensor(1.2809e-06)]


cfg: 35/36


Training: 100%|██████████| 1000/1000 [23:12<00:00,  1.39s/it, avg_train_batch_loss=tensor(3.7800e-06), avg_valid_batch_loss=tensor(7.7669e-06)]
